In [1]:
import pandas as pd
import json
import re
import os

from IPython.display import display

pd.set_option("display.max_colwidth", None)

In [2]:
data_path = os.path.join(os.path.dirname(os.path.abspath(".")), "data/raw/home_depot_data_1_2021_12.csv")
df = pd.read_csv(data_path, index_col=0)

In [5]:
test_cases_path = os.path.join(os.path.dirname(os.path.abspath(".")), "data/test_cases/test_cases.json")

with open(test_cases_path) as f:
    test_cases = json.load(f)

In [60]:
select_cols = ['title', 'description', 'brand']

In [61]:
df[select_cols].head(10)

,title,description,brand
index,,,
0,Men's 3X Large Carbon Heather Cotton/Polyester Rain Defender Paxton Heavyweight Hooded Zip-Front Sweatshirt,"This heavyweight, water-repellent hooded sweatshirt has a zip front for fast layering. ORIGINAL FIT. 13 oz., 75% cotton/25% polyester blend with Rain Defender durable water repellent. Attached, jersey-lined three-piece hood with drawcord closure. Antique-finish brass front zipper. Two front hand-warmer pockets have a hidden security pocket inside. Stretchable, spandex-reinforced rib-knit cuffs and waistband. Locker loop facilitates hanging.",Carhartt
1,Turmode 30 ft. RP TNC Female to RP TNC Male Adapter Cable,"If you need more length between your existing wireless device and Hi-Gain Antenna, this is the product for you. It's compatible with most Wi-Fi Antennas, so it is easy for you to extend your wireless network. Just replace your existing cable that runs between your wireless device and Antenna and you're ready to use your network with extended range.",Unbranded
2,Large Tapestry Bolster Bed,"Polyester cover resembling rich Italian tapestries wraps your pet in comfort and style. 100% recycled polyester high loft MemoryFiber fill keeps pets elevated off cold floors for relief on tired joints and pressure points. A great fit for pets up to 100 lbs. Zippered removable cover is machine washable in cold water only. To dry, use a delicate cycle or gentle setting with very low heat.",Carolina Pet Company
3,16-Gauge-Sinks Vessel Sink in White with Faucet,It features a rectangle shape. This vessel set is designed to be installed as a undermount vessel set. It is constructed with ceramic. This vessel set comes with an enamel glaze finish in White color. It is designed for a deck mount faucet.,Unbranded
4,Men's Crazy Horse 9'' Logger Boot - Steel Toe - Black Size 10.5(W),This 9 in. black full grain leather logger boot is Tough. With plain soft toe and rubber outsole it provides dependable traction with a comfortable fit. It is made with Goodyear Welt Construction so it is sturdy and secure.,Adtec
5,Mariana 6 ft. Multi-Color 3-Panel Screen Divider,"With robust structure and sophisticated canvas printing, this 3-panel screen is colorfully designed to be a great way to carve out space in your room or anywhere around the home. Not only will this elegant piece give you the needed exclusivity but it will also add inspiring style and beauty to your decor. This screen creatively features canvas print of a different and complementary images on each side. With its lightweight construction, our screen will definitely fill your space with inspiring visual flair for many-Years to come. 3-panel screen finished on both sides Canvas printing.",HomeRoots
6,5 gal. #650C-2 Powdery Mist Semi-Gloss Interior Paint,"BEHR PRO i300 Semi-Gloss Interior Paint has a sleek, radiant sheen appearance. This professional quality latex paint has superior hide and coverage, excellent sprayability, spray and back-roll, and superior touch-up. Use on properly prepared and primed drywall, concrete, masonry, wood and metal surfaces in both residential and commercial applications.",BEHR PRO
7,7/8 in. x 4-1/2 in. x 0.045 in. Metal and Stainless Cutting Wheel (50-Pack),DEWALT High Performance 0.045 in. Metal Cutting Wheels have a thin 0.045 in. cutting edge design for fast burr free cutting. They have a proprietary aluminum oxide grain combination for aggressive cutting action and the proprietary material mix ensures durable long life wheels. They have 2-full sheets of fiberglass for durability and user safety. These wheels are ideal for high performance cutting in all types of ferrous metals and stainless steel.,DEWALT
8,Ring Gold Bar Cart,This Ring Bar Cart is sure to make a statement in any room. This carts features a metal frame in gold and the two shelves are made of mirror with an antique finish. We offer a wide variety of furniture and decorative accessories to meet all of your decor needs.,Titan Lighting


In [62]:
df_search = df[select_cols]
df_search['combined_text'] = df_search['title']

In [63]:
#df_search['combined_text'] = df_search['title'].fillna('') + ' ' + df_search['description'].fillna('')
#df_search['combined_text'] = df_search['title']

## Clean data

The EDA analysis show that we need to perform the following cleaning steps:

- Remove non-ascii characters
- Remove special characters
- Lowercase text

In [64]:
def remove_non_ascii(text: str) -> str:
    if isinstance(text, str):
        return ''.join(char for char in text if ord(char) < 128)
    return text

In [65]:
def remove_special_characters(text: str) -> str:
    if isinstance(text, str):
        return ''.join(char for char in text if char.isalnum() or char.isspace())
    return text

In [66]:
def lowercase_text(text: str) -> str:
    if isinstance(text, str):
        return text.lower()
    return text

In [67]:
cleaning_steps = [
    remove_non_ascii,
    remove_special_characters,
    lowercase_text
]

result = df_search['combined_text'].copy()
for func in cleaning_steps:
    result = result.apply(func)
df_search['combined_text_clean'] = result

In [68]:
df_search.head()

,title,description,brand,combined_text,combined_text_clean
index,,,,,
0,Men's 3X Large Carbon Heather Cotton/Polyester Rain Defender Paxton Heavyweight Hooded Zip-Front Sweatshirt,"This heavyweight, water-repellent hooded sweatshirt has a zip front for fast layering. ORIGINAL FIT. 13 oz., 75% cotton/25% polyester blend with Rain Defender durable water repellent. Attached, jersey-lined three-piece hood with drawcord closure. Antique-finish brass front zipper. Two front hand-warmer pockets have a hidden security pocket inside. Stretchable, spandex-reinforced rib-knit cuffs and waistband. Locker loop facilitates hanging.",Carhartt,Men's 3X Large Carbon Heather Cotton/Polyester Rain Defender Paxton Heavyweight Hooded Zip-Front Sweatshirt,mens 3x large carbon heather cottonpolyester rain defender paxton heavyweight hooded zipfront sweatshirt
1,Turmode 30 ft. RP TNC Female to RP TNC Male Adapter Cable,"If you need more length between your existing wireless device and Hi-Gain Antenna, this is the product for you. It's compatible with most Wi-Fi Antennas, so it is easy for you to extend your wireless network. Just replace your existing cable that runs between your wireless device and Antenna and you're ready to use your network with extended range.",Unbranded,Turmode 30 ft. RP TNC Female to RP TNC Male Adapter Cable,turmode 30 ft rp tnc female to rp tnc male adapter cable
2,Large Tapestry Bolster Bed,"Polyester cover resembling rich Italian tapestries wraps your pet in comfort and style. 100% recycled polyester high loft MemoryFiber fill keeps pets elevated off cold floors for relief on tired joints and pressure points. A great fit for pets up to 100 lbs. Zippered removable cover is machine washable in cold water only. To dry, use a delicate cycle or gentle setting with very low heat.",Carolina Pet Company,Large Tapestry Bolster Bed,large tapestry bolster bed
3,16-Gauge-Sinks Vessel Sink in White with Faucet,It features a rectangle shape. This vessel set is designed to be installed as a undermount vessel set. It is constructed with ceramic. This vessel set comes with an enamel glaze finish in White color. It is designed for a deck mount faucet.,Unbranded,16-Gauge-Sinks Vessel Sink in White with Faucet,16gaugesinks vessel sink in white with faucet
4,Men's Crazy Horse 9'' Logger Boot - Steel Toe - Black Size 10.5(W),This 9 in. black full grain leather logger boot is Tough. With plain soft toe and rubber outsole it provides dependable traction with a comfortable fit. It is made with Goodyear Welt Construction so it is sturdy and secure.,Adtec,Men's Crazy Horse 9'' Logger Boot - Steel Toe - Black Size 10.5(W),mens crazy horse 9 logger boot steel toe black size 105w


## Keyword search engine

### Preprocessing text

Before creating BoW vectors, we need to preprocess the data.

In [69]:
def remove_stop_words(text: str) -> str:
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))
    if isinstance(text, str):
        return ' '.join(word for word in text.split() if word not in stop_words)
    return text

In [70]:
def stemm_words(text: str) -> str:
    from nltk.stem import PorterStemmer
    stemmer = PorterStemmer()
    if isinstance(text, str):
        return ' '.join(stemmer.stem(word) for word in text.split())
    return text

In [71]:
preprocessing_steps = [
    remove_stop_words,
    stemm_words
]

result = df_search['combined_text_clean'].copy()
for func in preprocessing_steps:
    result = result.apply(func)
    
df_search['combined_text_clean_preprocessed'] = result

In [72]:
def get_sparse_vectorizer(df: pd.DataFrame, column_name: str):
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(df[column_name])
    return tfidf_matrix, vectorizer


In [73]:
tfidf_matrix, vectorizer = get_sparse_vectorizer(df_search, 'combined_text_clean')


In [74]:
from sklearn.metrics.pairwise import cosine_similarity

def keyword_search_engine(query, df, tfidf_matrix, vectorizer, top_n=7):
    # 1. Transform the user's query using the already fitted vectorizer
    query_vec = vectorizer.transform([query])
    
    # 2. Calculate cosine similarity between the query and all documents
    cosine_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # 3. Add the scores to a copy of the dataframe
    results_df = df.copy()
    results_df['similarity_score'] = cosine_scores
    
    # 4. Filter for documents that have a score greater than 0 (relevant matches)
    results_df = results_df[results_df['similarity_score'] > 0]
    
    # 5. Sort by similarity score in descending order
    results_df = results_df.sort_values(by='similarity_score', ascending=False)
    
    # 6. Return the top N results
    return results_df.head(top_n)

In [75]:
def keyword_preprocess_query(query: str) -> str:
    for func in cleaning_steps + preprocessing_steps:
        query = func(query)
    return query

In [76]:
def get_keyword_search_results(user_query: str, df: pd.DataFrame, tfidf_matrix, vectorizer, top_n=10):
    preprocessed_query = keyword_preprocess_query(user_query)
    return keyword_search_engine(preprocessed_query, df, tfidf_matrix, vectorizer, top_n)

In [77]:
def print_search_results(df_search, user_query):
    print(f"Search results for: '{user_query}'\n")
    for _, row in df_search.iterrows():
        print(f"Title: {row['title']}")
        print(f"Description: {row['description']}")
        print(f"Score: {row['similarity_score']:.4f}\n")

In [78]:
def compute_search_metrics(result_titles, expected_titles):
    found = set(result_titles) & set(expected_titles)
    recall = len(found) / len(expected_titles) if expected_titles else 0
    return recall, found

def display_test_results(search_type, user_query, preprocessed_query, recall, expected_titles, found):
    print(f"{search_type:10} | user_query: {user_query:45} | preprocessed: {preprocessed_query:45} | recall: {recall:.2%}")
    comparison_df = pd.DataFrame({
        'Expected Titles': expected_titles,
        'Found': [title in found for title in expected_titles]
    })
    display(comparison_df.head(len(expected_titles)))

def test_keyword_search_engine(test_cases, search_type, df_search, tfidf_matrix, vectorizer):
    for test in test_cases:
        if test['search_type'] == search_type:
            user_query = test['keyword']
            results_df = get_keyword_search_results(user_query, df_search, tfidf_matrix, vectorizer)
            preprocessed_query = keyword_preprocess_query(user_query)
            result_titles = results_df['title'].tolist()
            expected_titles = [result['title'] for result in test['expected_results']]
            
            recall, found = compute_search_metrics(result_titles, expected_titles)
            display_test_results(search_type, user_query, preprocessed_query, recall, expected_titles, found)

In [79]:
search_type = "keyword"
test_keyword_search_engine(test_cases, search_type, df_search, tfidf_matrix, vectorizer)

keyword    | user_query: steel toe work boots                          | preprocessed: steel toe work boot                           | recall: 90.91%


,Expected Titles,Found
0,Men's Crazy Horse 9'' Logger Boot - Steel Toe - Black Size 10.5(W),True
1,Men's MobiLite Waterproof Oil-Resistant 8 in. Work Boot - Steel Toe - Brown - Size - 11(M),True
2,Men's Titanium Waterproof Work Boots - Steel Toe - Brown Size 8.5(W),True
3,Men's Floorhand Waterproof Wellington Work Boots - Steel Toe - Brown Size 10.5(M),True
4,Men's Waterproof Logger Boot - Steel Toe - Chocolate Size 10(W),True
5,Men's Force Slip Resistant Athletic-Nano Waterproof 6 in. Work Boots - Composite Toe - Brown Size 11(W),False
6,Men's Ground Force Waterproof Wellington Work Boots - Composite Toe - Brown Size 12(M),True
7,Men's Rugged Flex Waterproof 8'' Work Boots - Composite Toe - Brown Size 11(M),True
8,Men's McKinney Chelsea Waterproof Boot - Composite Toe - Black Size 13(W),True
9,Women's 8 in. Work Boots - Soft Toe - Black/Dark Cherry Size 9.5(M),True


keyword    | user_query: bathroom vanity mirror                        | preprocessed: bathroom vaniti mirror                        | recall: 23.81%


,Expected Titles,Found
0,33 in. W x 33 in. H (L2) Framed Square Deluxe Glass Bathroom Vanity Mirror in Weathered Wood,True
1,Blonde 32 in. W x 55 in. H Framed Rectangular Bathroom Vanity Mirror in Light Brown,True
2,31.6 in. W x 18.1 in. D x 32.9 in. H Complete Bathroom Vanity Unit in Anthracite with Mirror,True
3,Gela 60 in. W x 22 in. D x 35 in. H Vanity in Grey with Marble Vanity Top in White with Basin and Mirror,False
4,Venice 60 in. W x 18 in. D Bath Vanity in Navy with Marble Vanity Top in White with White Basin and Mirror,False
5,Daria 48 in. W x 22 in. D Single Vanity in Dark Espresso with Cultured Marble Vanity Top in White with Basin and Mirror,False
6,Caroline 72 in. W Bath Vanity in Espresso with Marble Vanity Top in White with Round Basin and Mirror,False
7,Caroline 36 in. W x 22 in. D x 35 in. H Single Sink Bath Vanity in Espresso with Quartz Top and Mirror,False
8,Caroline Avenue 72 in. W x 22 in. D x 35 in. H Double Sink Bath Vanity in Gray with Quartz Top and Mirror,False
9,Miranda 48 in. W Single Bath Vanity in Dark Blue with Solid Surface Vanity Top in White with White Basin and Mirror,False


## Semantic search engine

In [80]:
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

In [81]:
documents = df_search['title'].tolist()

In [82]:
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
 
print("Encoding documents (this may take a few seconds)...")
document_embeddings = model.encode(documents, show_progress_bar=True)
 
print(f"Created {document_embeddings.shape[0]} embeddings.")
print(f"Embedding dimension: {document_embeddings.shape[1]}")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7293.09it/s]


Encoding documents (this may take a few seconds)...


Batches: 100%|██████████| 80/80 [00:01<00:00, 42.75it/s]

Created 2551 embeddings.
Embedding dimension: 384


In [83]:
print("Creating vector index...")
semantic_search_engine = NearestNeighbors(n_neighbors=5, metric="cosine")
 
semantic_search_engine.fit(document_embeddings)
print("Vector indexis ready!")

Creating vector index...
Vector indexis ready!


In [84]:
def semantic_search(query, top_k=10):
    # Embed the incoming search query
    query_embedding = model.encode([query])
 
    # Retrieve the closest matches
    distances, indices = semantic_search_engine.kneighbors(query_embedding, n_neighbors=top_k)
 
        
    results_df = [documents[int(indices[0][i])] for i in range(top_k)]
    return results_df

In [85]:
def test_semantic_search_engine(test_cases):
    for test in test_cases:
        if test['search_type'] == "semantic":
            user_query = test['keyword']
            results_df = semantic_search(user_query)
            expected_titles = [result['title'] for result in test['expected_results']]
            
            recall, found = compute_search_metrics(results_df, expected_titles)
            display_test_results("semantic", user_query, "", recall, expected_titles, found)

In [86]:
test_semantic_search_engine(test_cases)

semantic   | user_query: waterproof jacket for cold weather            | preprocessed:                                               | recall: 100.00%


,Expected Titles,Found
0,Men's 3X Large Carbon Heather Cotton/Polyester Rain Defender Paxton Heavyweight Hooded Zip-Front Sweatshirt,True
1,Men's Small Dark Navy Cotton/Nylon FR Full Swing Quick Duck Coat,True
2,Women's X-Large Black 7.2-Volt Lithium-Ion Heated Down Vest with 90% Down Insulation and (1) 5.2 Ah Battery Pack,True
3,Men's Small Dark Navy Modacrylic/Lyocell/Aramid Fleece FR HW Zip Front Sweatshirt,True


semantic   | user_query: something to keep my dog from escaping the yard | preprocessed:                                               | recall: 50.00%


,Expected Titles,Found
0,Dig Defence Large Animal Dig Barrier (25-Pack),True
1,3 ft. x 48 ft. White Plastic Wire Mesh Fence Panel/Enclosure Kit with Gate Insert- Hard Surface,False
2,Athens 4 ft. W x 4 ft. H Gloss Black Aluminum Pressed Spear Design Fence Gate,False
3,4 ft. H x 25 ft. W Black Fence Outdoor Privacy Screen with Black Edge Bindings and Grommets,False
4,4 ft. x 144 ft. Grey Privacy Fence Screen HDPE Mesh Windscreen with Reinforced Grommets for Garden Fence (Custom Size),False
5,3 ft. x 72 ft. Beige Privacy Fence Screen HDPE Mesh Screen with Reinforced Grommets for Garden Fence (Custom Size),True
6,4 ft. x 77 ft. Green Privacy Fence Screen HDPE Mesh Windscreen with Reinforced Grommets for Garden Fence (Custom Size),False
7,8 ft. x 104 ft. Green Privacy Fence Screen HDPE Mesh Netting with Reinforced Grommets for Garden Fence (Custom Size),True
8,5 ft. x 161 ft. Brown Privacy Fence Screen Mesh Cover Screen with Reinforced Grommets for Garden Fence (Custom Size),True
9,8 ft. x 46 ft. White Privacy Fence Screen Mesh Cover Screen with Reinforced Grommets for Garden Fence (Custom Size),True


## Hybrid search

A semantic search engine combines keyword search (BM25) with semantic search (dense vector embeddings) and merge their scores using Reciprocal Rank Fusion (RRF).

For this implementation, we're using BM25 for keyword search (instead of TF-IDF with cosine similarity).

In [87]:
keyword_preprocess_query("cordless drill with battery")

'cordless drill batteri'

In [88]:
from rank_bm25 import BM25Okapi

corpus = df_search['combined_text_clean_preprocessed'].to_list()
tokenized_corpus = [doc.split(" ") for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

In [89]:
def search_bm25(query, top_k=10):
    preprocessed_query = keyword_preprocess_query(query)
    tokenized_query = preprocessed_query.split()
    
    # Getting scores (lexical relevance to the query) for all documents
    scores = bm25.get_scores(tokenized_query)
    
    # Ranking documents by score
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return ranked_indices[:top_k], scores

In [90]:
ranked_indices, scores = search_bm25("steel toe work boots")
df_search.iloc[ranked_indices]['title'].head()

index
2228                          Men's Titanium Waterproof Work Boots - Steel Toe - Brown Size 8.5(W)
2377             Men's Floorhand Waterproof Wellington Work Boots - Steel Toe - Brown Size 10.5(M)
680     Men's MobiLite Waterproof Oil-Resistant 8 in. Work Boot - Steel Toe - Brown - Size - 11(M)
617                            Women's 8 in. Work Boots - Soft Toe - Black/Dark Cherry Size 9.5(M)
1392        Men's Ground Force Waterproof Wellington Work Boots - Composite Toe - Brown Size 12(M)
Name: title, dtype: str

In [91]:
from sentence_transformers import util
import torch
doc_embeddings = model.encode(documents, convert_to_tensor=True)
 
def search_semantic(query, top_k=10):
    # Embedding the user's query into a vector
    query_embedding = model.encode(query, convert_to_tensor=True)
    
    # Calculating cosine similarity between the query and all documents
    cosine_scores = util.cos_sim(query_embedding, doc_embeddings)[0]
    
    # Ranking documents by similarity
    ranked_indices = torch.argsort(cosine_scores, descending=True).tolist()
    return ranked_indices[:top_k], cosine_scores.tolist()


In [92]:
ranked_indices, scores = search_semantic("waterproof jacket for cold weather")
df_search.iloc[ranked_indices]['title'].head()

index
0       Men's 3X Large Carbon Heather Cotton/Polyester Rain Defender Paxton Heavyweight Hooded Zip-Front Sweatshirt
2199                              Men's Small Dark Navy Modacrylic/Lyocell/Aramid Fleece FR HW Zip Front Sweatshirt
804                                                Men's Small Dark Navy Cotton/Nylon FR Full Swing Quick Duck Coat
680                      Men's MobiLite Waterproof Oil-Resistant 8 in. Work Boot - Steel Toe - Brown - Size - 11(M)
2257                                  Men's Homeland Waterproof 8 inch Lace Up Work Boots - Soft Toe - Brown 10 (W)
Name: title, dtype: str

In [93]:
def hybrid_search(query, top_k=10):
    # 1. Obtaining the two standalone search rankings
    bm25_ranks, _ = search_bm25(query, top_k=len(documents))
    semantic_ranks, _ = search_semantic(query, top_k=len(documents))
    
    # 2. Applying RRF formula: RRF_score = 1 / (k + rank)
    rrf_scores = {i: 0.0 for i in range(len(documents))}
    k_constant = 60  # The value of 60 is a standard academic convention
    
    # Adding RRF scores from BM25
    for rank, doc_idx in enumerate(bm25_ranks):
        rrf_scores[doc_idx] += 1.0 / (k_constant + rank + 1)
        
    # Adding RRF scores from semantic search
    for rank, doc_idx in enumerate(semantic_ranks):
        rrf_scores[doc_idx] += 1.0 / (k_constant + rank + 1)
    
    # 3. Sorting documents by their final fused RRF score
    final_ranked_indices = sorted(rrf_scores.keys(), key=lambda idx: rrf_scores[idx], reverse=True)
    
    return final_ranked_indices[:top_k], rrf_scores

In [95]:
ranked_indices, scores = hybrid_search("blue paint for bedroom walls")
df_search.iloc[ranked_indices]['title'].head(n=10)

index
1756                                                   "Blue" by Peter Morneau Framed Abstract Wall Art Print 30 in. x 30 in.
330                                                       24 in. x 36 in. "Black & Grey & Blue I" by Studio W Framed Wall Art
869                                                                              Americana 2 oz. Victorian Blue Acrylic Paint
295                                                                           5 gal. #HDGB52 Village Blue Flat Exterior Paint
1831                                                                 5-gal. #HDGV23 Blue Silk Semi-Gloss Latex Exterior Paint
342                                                                      1 gal. #ICC-99 Alluring Blue Eggshell Interior Paint
1994                                                     1 gal. #T15-8 Elusive Blue Semi-Gloss Enamel Exterior Paint & Primer
1699                                                               1 gal. #HDGB46D Hidden Harbor Blue Eggshell I